In [17]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np

# === Read the AGE results file ===
df = pd.read_csv("../results/age_results_year.tsv", sep="\t")

# Only keep the model of interest
df = df[df["model"] == "distilbert-base-uncased"]

# All age pairs present in the dataset (consistent ordering)
age_pairs_to_plot = sorted(df["age_pair"].unique())

bar_width = 0.25

# === One color per SHORT AGE GROUP ===

age_color_map = {
    "young": "#8DA0CB",
    "middle": "#FC8D62",
    "older": "#66C2A5",
}


# Mapping from full phrase → short name
short_name = {
    "young adult": "young",
    "middle-aged adult": "middle",
    "older adult": "older",
}

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(8, 5), sharey=True)
axes = axes.flatten()

temp = ["M1", "M9", "D1", "D4"]
crimes = ["Murder", "Fraud", "Distribution of Drug", "Possession of Drug"]
level = ["Serious", "Light", "Serious", "Light"]

for plot_idx, ax in enumerate(axes):
    sev = temp[plot_idx]

    df_sev = df[df["severity"] == sev]

    if df_sev.empty:
        ax.set_visible(False)
        continue

    # Aggregate
    bar_df = df_sev.groupby(["age_pair", "years"])["acc"].mean().reset_index()
    bar_data = bar_df.pivot(index="years", columns="age_pair", values="acc")

    # Match global ordering
    age_pairs = [ap for ap in age_pairs_to_plot if ap in bar_data.columns]
    bar_data = bar_data[age_pairs]

    x_positions_for_years = np.arange(len(bar_data.index))
    num_pairs = len(age_pairs)

    for i, age_pair in enumerate(age_pairs):

        # Split full pair
        age1_full, age2_full = age_pair.split(" vs ")

        # Convert to short labels
        age1 = short_name[age1_full]
        age2 = short_name[age2_full]

        acc_values = bar_data[age_pair]  # preference for age1
        comp_values = 1 - acc_values  # preference for age2

        color_bottom = age_color_map.get(age1, "gray")
        color_top = age_color_map.get(age2, "lightgray")

        bar_x_positions = (
            x_positions_for_years - (bar_width * num_pairs / 2) + (i + 0.5) * bar_width
        )

        # Bottom bar (first group)
        bars = ax.bar(
            bar_x_positions,
            acc_values,
            width=bar_width,
            color=color_bottom,
            edgecolor="White",
        )

        # Top bar (second group)
        con_bars = ax.bar(
            bar_x_positions,
            comp_values,
            width=bar_width,
            bottom=acc_values,
            alpha=1,
            color=color_top,
            edgecolor="White",
        )

        # Label bottom with first group (short name)
        bottom_labels = [age1] * len(acc_values)
        ax.bar_label(
            bars,
            labels=bottom_labels,
            label_type="center",
            color="white",
            fontsize=4,
            fontweight="bold",
        )

        # Label top with second group (short name)
        top_labels = [age2] * len(comp_values)
        ax.bar_label(
            con_bars,
            labels=top_labels,
            label_type="center",
            color="white",
            fontsize=4,
            fontweight="bold",
        )

    ax.set_xticks(x_positions_for_years)
    ax.set_xticklabels(bar_data.index)
    ax.set_xlabel("Sentence Length (Years)")
    ax.set_ylabel("Preference Proportion")
    ax.set_title(f"{level[plot_idx]} Crime e.g. ({crimes[plot_idx]})")
    ax.set_ylim(0, 1.1)
    ax.axhline(0.5, linestyle="--", color="gray", linewidth=1)

# Legend for short groups
handles = [
    plt.Line2D(
        [0],
        [0],
        marker="s",
        linestyle="",
        markerfacecolor=color,
        markeredgecolor="black",
        label=age,
    )
    for age, color in age_color_map.items()
]
fig.legend(handles=handles, loc="upper right", title="Age group")

plt.suptitle(
    "Distilbert",
    fontsize=16,
    y=1.0,
)

plt.tight_layout(rect=[0, 0.03, 0.97, 0.96])

os.makedirs("../assets", exist_ok=True)
plt.savefig("../assets/Distilbert_Uncased_age.png", dpi=300, bbox_inches="tight")
plt.close()

print("2x2 subplot grid for age groups saved!")

2x2 subplot grid for age groups saved!


In [18]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np

# === Read the AGE results file ===
df = pd.read_csv("../results/age_results_year.tsv", sep="\t")

# Only keep GPT-2 model
df = df[df["model"] == "gpt2"]

# All age pairs present in the dataset (consistent ordering)
age_pairs_to_plot = sorted(df["age_pair"].unique())

bar_width = 0.25

# === One color per SHORT AGE GROUP ===
age_color_map = {
    "young": "#8DA0CB",
    "middle": "#FC8D62",
    "older": "#66C2A5",
}

# Map long → short labels
short_name = {
    "young adult": "young",
    "middle-aged adult": "middle",
    "older adult": "older",
}

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(8, 5), sharey=True)
axes = axes.flatten()

temp = ["M1", "M9", "D1", "D4"]
crimes = ["Murder", "Fraud", "Distribution of Drug", "Possession of Drug"]
level = ["Serious", "Light", "Serious", "Light"]

for plot_idx, ax in enumerate(axes):
    sev = temp[plot_idx]

    df_sev = df[df["severity"] == sev]

    if df_sev.empty:
        ax.set_visible(False)
        continue

    # Aggregate
    bar_df = df_sev.groupby(["age_pair", "years"])["acc"].mean().reset_index()
    bar_data = bar_df.pivot(index="years", columns="age_pair", values="acc")

    # Keep a consistent ordering
    age_pairs = [ap for ap in age_pairs_to_plot if ap in bar_data.columns]
    bar_data = bar_data[age_pairs]

    x_positions_for_years = np.arange(len(bar_data.index))
    num_pairs = len(age_pairs)

    for i, age_pair in enumerate(age_pairs):

        # Extract long form groups
        age1_full, age2_full = age_pair.split(" vs ")

        # Convert to short names
        age1 = short_name[age1_full]
        age2 = short_name[age2_full]

        acc_values = bar_data[age_pair]  # preference for first group
        comp_values = 1 - acc_values  # preference for second group

        color_bottom = age_color_map.get(age1, "gray")
        color_top = age_color_map.get(age2, "lightgray")

        bar_x_positions = (
            x_positions_for_years - (bar_width * num_pairs / 2) + (i + 0.5) * bar_width
        )

        # Bottom bar
        bars = ax.bar(
            bar_x_positions,
            acc_values,
            width=bar_width,
            color=color_bottom,
            edgecolor="white",
        )

        # Top bar
        con_bars = ax.bar(
            bar_x_positions,
            comp_values,
            width=bar_width,
            bottom=acc_values,
            alpha=1,
            color=color_top,
            edgecolor="white",
        )

        # Label bottom (first group)
        ax.bar_label(
            bars,
            labels=[age1] * len(acc_values),
            label_type="center",
            color="white",
            fontsize=4,
            fontweight="bold",
        )

        # Label top (second group)
        ax.bar_label(
            con_bars,
            labels=[age2] * len(comp_values),
            label_type="center",
            color="white",
            fontsize=4,
            fontweight="bold",
        )

    ax.set_xticks(x_positions_for_years)
    ax.set_xticklabels(bar_data.index)
    ax.set_xlabel("Sentence Length (Years)")
    ax.set_ylabel("Preference Proportion")
    ax.set_title(f"{level[plot_idx]} Crime e.g. ({crimes[plot_idx]})")
    ax.set_ylim(0, 1.1)
    ax.axhline(0.5, linestyle="--", color="gray", linewidth=1)

# Legend for short age groups
handles = [
    plt.Line2D(
        [0],
        [0],
        marker="s",
        linestyle="",
        markerfacecolor=color,
        markeredgecolor="black",
        label=age,
    )
    for age, color in age_color_map.items()
]
fig.legend(handles=handles, loc="upper right", title="Age group")

plt.suptitle(
    "GPT-2",
    fontsize=16,
    y=1.0,
)

plt.tight_layout(rect=[0, 0.03, 0.97, 0.96])

os.makedirs("../assets", exist_ok=True)
plt.savefig("../assets/GPT2_age.png", dpi=300, bbox_inches="tight")
plt.close()

print("2x2 subplot grid for age groups (GPT-2) saved!")

2x2 subplot grid for age groups (GPT-2) saved!


In [19]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np

# === Read the AGE results file ===
df = pd.read_csv("../results/age_results_year.tsv", sep="\t")

# Only keep SmolLM2-135M model
df = df[df["model"] == "HuggingFaceTB/SmolLM2-135M"]

# All age pairs present in the dataset (consistent ordering)
age_pairs_to_plot = sorted(df["age_pair"].unique())

bar_width = 0.25

# === One color per SHORT AGE GROUP ===
age_color_map = {
    "young": "#8DA0CB",
    "middle": "#FC8D62",
    "older": "#66C2A5",
}

# Map long → short labels
short_name = {
    "young adult": "young",
    "middle-aged adult": "middle",
    "older adult": "older",
}

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(8, 5), sharey=True)
axes = axes.flatten()

temp = ["M1", "M9", "D1", "D4"]
crimes = ["Murder", "Fraud", "Distribution of Drug", "Possession of Drug"]
level = ["Serious", "Light", "Serious", "Light"]

for plot_idx, ax in enumerate(axes):
    sev = temp[plot_idx]

    df_sev = df[df["severity"] == sev]

    if df_sev.empty:
        ax.set_visible(False)
        continue

    # Aggregate
    bar_df = df_sev.groupby(["age_pair", "years"])["acc"].mean().reset_index()
    bar_data = bar_df.pivot(index="years", columns="age_pair", values="acc")

    # Keep a consistent ordering
    age_pairs = [ap for ap in age_pairs_to_plot if ap in bar_data.columns]
    bar_data = bar_data[age_pairs]

    x_positions_for_years = np.arange(len(bar_data.index))
    num_pairs = len(age_pairs)

    for i, age_pair in enumerate(age_pairs):

        # Extract long form groups
        age1_full, age2_full = age_pair.split(" vs ")

        # Convert to short names
        age1 = short_name[age1_full]
        age2 = short_name[age2_full]

        acc_values = bar_data[age_pair]  # preference for first group
        comp_values = 1 - acc_values  # preference for second group

        color_bottom = age_color_map.get(age1, "gray")
        color_top = age_color_map.get(age2, "lightgray")

        bar_x_positions = (
            x_positions_for_years - (bar_width * num_pairs / 2) + (i + 0.5) * bar_width
        )

        # Bottom bar
        bars = ax.bar(
            bar_x_positions,
            acc_values,
            width=bar_width,
            color=color_bottom,
            edgecolor="white",
        )

        # Top bar
        con_bars = ax.bar(
            bar_x_positions,
            comp_values,
            width=bar_width,
            bottom=acc_values,
            alpha=1,
            color=color_top,
            edgecolor="white",
        )

        # Label bottom (first group)
        ax.bar_label(
            bars,
            labels=[age1] * len(acc_values),
            label_type="center",
            color="white",
            fontsize=4,
            fontweight="bold",
        )

        # Label top (second group)
        ax.bar_label(
            con_bars,
            labels=[age2] * len(comp_values),
            label_type="center",
            color="white",
            fontsize=4,
            fontweight="bold",
        )

    ax.set_xticks(x_positions_for_years)
    ax.set_xticklabels(bar_data.index)
    ax.set_xlabel("Sentence Length (Years)")
    ax.set_ylabel("Preference Proportion")
    ax.set_title(f"{level[plot_idx]} Crime e.g. ({crimes[plot_idx]})")
    ax.set_ylim(0, 1.1)
    ax.axhline(0.5, linestyle="--", color="gray", linewidth=1)

# Legend for short age groups
handles = [
    plt.Line2D(
        [0],
        [0],
        marker="s",
        linestyle="",
        markerfacecolor=color,
        markeredgecolor="black",
        label=age,
    )
    for age, color in age_color_map.items()
]
fig.legend(handles=handles, loc="upper right", title="Age group")

plt.suptitle(
    "SmolLM2-135M",
    fontsize=16,
    y=1.0,
)

plt.tight_layout(rect=[0, 0.03, 0.97, 0.96])

os.makedirs("../assets", exist_ok=True)
plt.savefig("../assets/SmolLM2-135M_age.png", dpi=300, bbox_inches="tight")
plt.close()

print("2x2 subplot grid for age groups (SmolLM2-135M) saved!")

2x2 subplot grid for age groups (SmolLM2-135M) saved!


In [20]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np

# === Read the AGE results file ===
df = pd.read_csv("../results/age_results_year.tsv", sep="\t")

# Only keep SmolLM2-135M model
df = df[df["model"] == "HuggingFaceTB/SmolLM2-135M"]

# All age pairs present in the dataset (consistent ordering)
age_pairs_to_plot = sorted(df["age_pair"].unique())

bar_width = 0.25

# === One color per SHORT AGE GROUP ===
age_color_map = {
    "young": "#8DA0CB",
    "middle": "#FC8D62",
    "older": "#66C2A5",
}

# Map long → short labels
short_name = {
    "young adult": "young",
    "middle-aged adult": "middle",
    "older adult": "older",
}

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(8, 5), sharey=True)
axes = axes.flatten()

temp = ["M1", "M9", "D1", "D4"]
crimes = ["Murder", "Fraud", "Distribution of Drug", "Possession of Drug"]
level = ["Serious", "Light", "Serious", "Light"]

for plot_idx, ax in enumerate(axes):
    sev = temp[plot_idx]

    df_sev = df[df["severity"] == sev]

    if df_sev.empty:
        ax.set_visible(False)
        continue

    # Aggregate
    bar_df = df_sev.groupby(["age_pair", "years"])["acc"].mean().reset_index()
    bar_data = bar_df.pivot(index="years", columns="age_pair", values="acc")

    # Keep a consistent ordering
    age_pairs = [ap for ap in age_pairs_to_plot if ap in bar_data.columns]
    bar_data = bar_data[age_pairs]

    x_positions_for_years = np.arange(len(bar_data.index))
    num_pairs = len(age_pairs)

    for i, age_pair in enumerate(age_pairs):

        # Extract long form groups
        age1_full, age2_full = age_pair.split(" vs ")

        # Convert to short names
        age1 = short_name[age1_full]
        age2 = short_name[age2_full]

        acc_values = bar_data[age_pair]  # preference for first group
        comp_values = 1 - acc_values  # preference for second group

        color_bottom = age_color_map.get(age1, "gray")
        color_top = age_color_map.get(age2, "lightgray")

        bar_x_positions = (
            x_positions_for_years - (bar_width * num_pairs / 2) + (i + 0.5) * bar_width
        )

        # Bottom bar
        bars = ax.bar(
            bar_x_positions,
            acc_values,
            width=bar_width,
            color=color_bottom,
            edgecolor="white",
        )

        # Top bar
        con_bars = ax.bar(
            bar_x_positions,
            comp_values,
            width=bar_width,
            bottom=acc_values,
            alpha=1,
            color=color_top,
            edgecolor="white",
        )

        # Label bottom (first group)
        ax.bar_label(
            bars,
            labels=[age1] * len(acc_values),
            label_type="center",
            color="white",
            fontsize=4,
            fontweight="bold",
        )

        # Label top (second group)
        ax.bar_label(
            con_bars,
            labels=[age2] * len(comp_values),
            label_type="center",
            color="white",
            fontsize=4,
            fontweight="bold",
        )

    ax.set_xticks(x_positions_for_years)
    ax.set_xticklabels(bar_data.index)
    ax.set_xlabel("Sentence Length (Years)")
    ax.set_ylabel("Preference Proportion")
    ax.set_title(f"{level[plot_idx]} Crime e.g. ({crimes[plot_idx]})")
    ax.set_ylim(0, 1.1)
    ax.axhline(0.5, linestyle="--", color="gray", linewidth=1)

# Legend for short age groups
handles = [
    plt.Line2D(
        [0],
        [0],
        marker="s",
        linestyle="",
        markerfacecolor=color,
        markeredgecolor="black",
        label=age,
    )
    for age, color in age_color_map.items()
]
fig.legend(handles=handles, loc="upper right", title="Age group")

plt.suptitle(
    "SmolLM2-135M",
    fontsize=16,
    y=1.0,
)

plt.tight_layout(rect=[0, 0.03, 0.97, 0.96])

os.makedirs("../assets", exist_ok=True)
plt.savefig("../assets/SmolLM2-135M_age.png", dpi=300, bbox_inches="tight")
plt.close()

print("2x2 subplot grid for age groups (SmolLM2-135M) saved!")

2x2 subplot grid for age groups (SmolLM2-135M) saved!


In [21]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np

# === Read the AGE results file ===
df = pd.read_csv("../results/age_results_year.tsv", sep="\t")

# Only keep SmolLM2-135M model
df = df[df["model"] == "distilroberta-base"]

# All age pairs present in the dataset (consistent ordering)
age_pairs_to_plot = sorted(df["age_pair"].unique())

bar_width = 0.25

# === One color per SHORT AGE GROUP ===
age_color_map = {
    "young": "#8DA0CB",
    "middle": "#FC8D62",
    "older": "#66C2A5",
}

# Map long → short labels
short_name = {
    "young adult": "young",
    "middle-aged adult": "middle",
    "older adult": "older",
}

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(8, 5), sharey=True)
axes = axes.flatten()

temp = ["M1", "M9", "D1", "D4"]
crimes = ["Murder", "Fraud", "Distribution of Drug", "Possession of Drug"]
level = ["Serious", "Light", "Serious", "Light"]

for plot_idx, ax in enumerate(axes):
    sev = temp[plot_idx]

    df_sev = df[df["severity"] == sev]

    if df_sev.empty:
        ax.set_visible(False)
        continue

    # Aggregate
    bar_df = df_sev.groupby(["age_pair", "years"])["acc"].mean().reset_index()
    bar_data = bar_df.pivot(index="years", columns="age_pair", values="acc")

    # Keep a consistent ordering
    age_pairs = [ap for ap in age_pairs_to_plot if ap in bar_data.columns]
    bar_data = bar_data[age_pairs]

    x_positions_for_years = np.arange(len(bar_data.index))
    num_pairs = len(age_pairs)

    for i, age_pair in enumerate(age_pairs):

        # Extract long form groups
        age1_full, age2_full = age_pair.split(" vs ")

        # Convert to short names
        age1 = short_name[age1_full]
        age2 = short_name[age2_full]

        acc_values = bar_data[age_pair]  # preference for first group
        comp_values = 1 - acc_values  # preference for second group

        color_bottom = age_color_map.get(age1, "gray")
        color_top = age_color_map.get(age2, "lightgray")

        bar_x_positions = (
            x_positions_for_years - (bar_width * num_pairs / 2) + (i + 0.5) * bar_width
        )

        # Bottom bar
        bars = ax.bar(
            bar_x_positions,
            acc_values,
            width=bar_width,
            color=color_bottom,
            edgecolor="white",
        )

        # Top bar
        con_bars = ax.bar(
            bar_x_positions,
            comp_values,
            width=bar_width,
            bottom=acc_values,
            alpha=1,
            color=color_top,
            edgecolor="white",
        )

        # Label bottom (first group)
        ax.bar_label(
            bars,
            labels=[age1] * len(acc_values),
            label_type="center",
            color="white",
            fontsize=4,
            fontweight="bold",
        )

        # Label top (second group)
        ax.bar_label(
            con_bars,
            labels=[age2] * len(comp_values),
            label_type="center",
            color="white",
            fontsize=4,
            fontweight="bold",
        )

    ax.set_xticks(x_positions_for_years)
    ax.set_xticklabels(bar_data.index)
    ax.set_xlabel("Sentence Length (Years)")
    ax.set_ylabel("Preference Proportion")
    ax.set_title(f"{level[plot_idx]} Crime e.g. ({crimes[plot_idx]})")
    ax.set_ylim(0, 1.1)
    ax.axhline(0.5, linestyle="--", color="gray", linewidth=1)

# Legend for short age groups
handles = [
    plt.Line2D(
        [0],
        [0],
        marker="s",
        linestyle="",
        markerfacecolor=color,
        markeredgecolor="black",
        label=age,
    )
    for age, color in age_color_map.items()
]
fig.legend(handles=handles, loc="upper right", title="Age group")

plt.suptitle(
    "Distilroberta",
    fontsize=16,
    y=1.0,
)

plt.tight_layout(rect=[0, 0.03, 0.97, 0.96])

os.makedirs("../assets", exist_ok=True)
plt.savefig("../assets/Distilroberta_age.png", dpi=300, bbox_inches="tight")
plt.close()

print("2x2 subplot grid for age groups (distilroberta) saved!")

2x2 subplot grid for age groups (distilroberta) saved!
